# Domain D: Spatial Organization and Micro-Architecture

This notebook addresses two questions:
- **D1:** Is there spatial clustering of functionally similar neurons beyond what cell-type identity predicts?
- **D2:** Does cell-type composition vary across the columnar extent and relate to functional properties?

**Data:** Zarr multimodal stores with 3D cell coordinates (x_loc, y_loc, z_loc), cell-type labels, ΔF/F traces, and gene expression.

**Note:** Additional high-precision 3D centroids, soma volume (n_voxels), soma elongation (size_pc1/2/3_um), and orientation angle are available in the zarr multimodal stores under `morphology/mask_properties/`. See Domain H notebook for morphology-specific analyses.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, chi2_contingency
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr
from functions.tuning import compute_osi, preferred_orientation
from functions.analysis import morans_i

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# ── 3D coordinates ──
coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
print(f"Total cells: {len(obs)}")
print(f"Coordinate ranges: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], "
      f"y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}], "
      f"z=[{coords[:,2].min():.0f}, {coords[:,2].max():.0f}]")

## D1: Spatial Clustering of Functionally Similar Neurons

Compute Moran's I for orientation preference, signal/noise correlations vs pairwise distance, and test whether functional similarity at short range exceeds what cell-type identity alone predicts.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.1  Compute tuning vectors for orientation preference
# ══════════════════════════════════════════════════════════════════════

# Orientation tuning from contrast-context blocks
contrast_blocks = var['stim_block'].isin([0.0, 2.0])
ori_responses = np.zeros((adata.n_obs, len(orientations)))
for i, ori in enumerate(orientations):
    mask = contrast_blocks & (var['orientation'] == ori)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 0:
        ori_responses[:, i] = np.nanmean(X[:, trial_idx], axis=1)

pref_ori = np.array([preferred_orientation(ori_responses[i], orientations) for i in range(adata.n_obs)])
osi = np.array([compute_osi(ori_responses[i], orientations) for i in range(adata.n_obs)])

print(f"Preferred orientation range: {np.nanmin(pref_ori):.1f}° to {np.nanmax(pref_ori):.1f}°")
print(f"OSI range: {np.nanmin(osi):.3f} to {np.nanmax(osi):.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.2  Moran's I: Spatial autocorrelation of orientation preference
# ══════════════════════════════════════════════════════════════════════

# Compute Moran's I at multiple distance thresholds (per mouse to avoid cross-mouse artifacts)
distance_bins = [30, 50, 75, 100, 150, 200, 300]
mouse_ids = obs['mouse_id'].unique()

morans_results = []
for mouse in mouse_ids:
    mouse_mask = obs['mouse_id'].values == mouse
    mouse_coords = coords[mouse_mask]
    mouse_pref = pref_ori[mouse_mask]
    mouse_osi = osi[mouse_mask]
    
    # Use circular variable for orientation: cos(2*pref_ori) and sin(2*pref_ori)
    cos_pref = np.cos(np.deg2rad(2 * mouse_pref))
    sin_pref = np.sin(np.deg2rad(2 * mouse_pref))
    
    for d in distance_bins:
        I_cos = morans_i(cos_pref, mouse_coords, d)
        I_osi = morans_i(mouse_osi, mouse_coords, d)
        morans_results.append({'mouse': mouse, 'distance': d, 'measure': 'cos(2θ)', 'I': I_cos})
        morans_results.append({'mouse': mouse, 'distance': d, 'measure': 'OSI', 'I': I_osi})

morans_df = pd.DataFrame(morans_results)

# ── Permutation test for significance ──
n_perm = 200
print("\nPermutation test (200 shuffles):")
for mouse in mouse_ids:
    mouse_mask = obs['mouse_id'].values == mouse
    mouse_coords = coords[mouse_mask]
    mouse_pref = pref_ori[mouse_mask]
    cos_pref = np.cos(np.deg2rad(2 * mouse_pref))
    
    I_obs = morans_i(cos_pref, mouse_coords, 50)
    I_perm = []
    for _ in range(n_perm):
        shuffled = np.random.permutation(cos_pref)
        I_perm.append(morans_i(shuffled, mouse_coords, 50))
    I_perm = np.array(I_perm)
    p = np.mean(I_perm >= I_obs)
    print(f"  Mouse {mouse}: I_obs={I_obs:.4f}, p={p:.3f} (I_perm mean={np.mean(I_perm):.4f})")

# ── Visualization: Moran's I correlogram ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, measure in zip(axes, ['cos(2θ)', 'OSI']):
    sub = morans_df[morans_df['measure'] == measure]
    for mouse in mouse_ids:
        m_sub = sub[sub['mouse'] == mouse]
        ax.plot(m_sub['distance'], m_sub['I'], 'o-', label=f'Mouse {mouse}', alpha=0.7)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    ax.set_xlabel('Distance Threshold (µm)')
    ax.set_ylabel("Moran's I")
    ax.set_title(f"D1: Moran's I — {measure}", fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle("D1: Spatial Autocorrelation of Tuning Properties", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.3  Signal & Noise correlation vs pairwise distance
# ══════════════════════════════════════════════════════════════════════

# Use a single mouse to keep computation tractable
mouse_pick = mouse_ids[np.argmax([np.sum(obs['mouse_id'].values == m) for m in mouse_ids])]
m_mask = obs['mouse_id'].values == mouse_pick
m_X = X[m_mask]
m_coords = coords[m_mask]
m_ori_resp = ori_responses[m_mask]
m_subclass = obs['subclass_name'].values[m_mask]
n_m = m_mask.sum()
print(f"Computing correlations for mouse {mouse_pick} ({n_m} cells)")

# Signal correlation: correlation of trial-averaged tuning vectors
signal_corr = np.corrcoef(m_ori_resp)

# Noise correlation: trial-by-trial residuals
# For each condition (orientation), subtract the trial-averaged response, then correlate residuals
contrast_blocks_m = var['stim_block'].isin([0.0, 2.0])
residuals_all = []
for ori in orientations:
    mask = contrast_blocks_m & (var['orientation'] == ori)
    trial_idx = np.where(mask.values)[0]
    if len(trial_idx) > 5:
        trial_resp = m_X[:, trial_idx]  # cells x trials
        mean_resp = np.nanmean(trial_resp, axis=1, keepdims=True)
        residuals_all.append(trial_resp - mean_resp)

if residuals_all:
    residuals_cat = np.hstack(residuals_all)  # cells x total_trials
    noise_corr = np.corrcoef(residuals_cat)
else:
    noise_corr = np.full((n_m, n_m), np.nan)

# Pairwise distance
dist_mat = squareform(pdist(m_coords))

# ── Bin by distance and plot ──
distance_edges = np.arange(0, 400, 25)
distance_centers = (distance_edges[:-1] + distance_edges[1:]) / 2

# Get upper triangle indices
triu_i, triu_j = np.triu_indices(n_m, k=1)
pair_dist = dist_mat[triu_i, triu_j]
pair_signal = signal_corr[triu_i, triu_j]
pair_noise = noise_corr[triu_i, triu_j]
pair_same_type = m_subclass[triu_i] == m_subclass[triu_j]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, corr_vals, title in zip(axes[:2], [pair_signal, pair_noise],
                                  ['Signal Correlation', 'Noise Correlation']):
    mean_all, mean_same, mean_diff = [], [], []
    for b in range(len(distance_edges)-1):
        bin_mask = (pair_dist >= distance_edges[b]) & (pair_dist < distance_edges[b+1])
        valid = bin_mask & ~np.isnan(corr_vals)
        if valid.sum() > 10:
            mean_all.append(np.nanmean(corr_vals[valid]))
            mean_same.append(np.nanmean(corr_vals[valid & pair_same_type]))
            mean_diff.append(np.nanmean(corr_vals[valid & ~pair_same_type]))
        else:
            mean_all.append(np.nan)
            mean_same.append(np.nan)
            mean_diff.append(np.nan)
    
    ax.plot(distance_centers, mean_all, 'k-', linewidth=2, label='All pairs')
    ax.plot(distance_centers, mean_same, 'r-', linewidth=2, label='Same subclass')
    ax.plot(distance_centers, mean_diff, 'b-', linewidth=2, label='Different subclass')
    ax.set_xlabel('Pairwise Distance (µm)')
    ax.set_ylabel(title)
    ax.set_title(f'D1: {title} vs Distance', fontweight='bold')
    ax.legend(fontsize=8)
    ax.axhline(0, color='gray', ls='--', alpha=0.4)

# Residual signal correlation after regressing out cell-type identity
# (test if functional similarity beyond type depends on distance)
ax = axes[2]
# Within same-type pairs only
same_signal = pair_signal[pair_same_type & ~np.isnan(pair_signal)]
same_dist = pair_dist[pair_same_type & ~np.isnan(pair_signal)]
r, p = spearmanr(same_dist, same_signal)
ax.scatter(same_dist, same_signal, alpha=0.05, s=5, c='gray')
# Binned mean
for b in range(len(distance_edges)-1):
    bin_mask = (same_dist >= distance_edges[b]) & (same_dist < distance_edges[b+1])
    if bin_mask.sum() > 5:
        ax.scatter(distance_centers[b], np.mean(same_signal[bin_mask]), c='red', s=40, zorder=5)
ax.set_xlabel('Distance (µm)')
ax.set_ylabel('Signal Correlation')
ax.set_title(f'D1: Within-Type Signal Corr vs Dist (ρ={r:.3f}, p={p:.1e})', fontweight='bold')
ax.axhline(0, color='gray', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# D1.4  Spatial maps of orientation preference (per depth)
# ══════════════════════════════════════════════════════════════════════

# Color by preferred orientation using a circular colormap
from matplotlib.colors import hsv_to_rgb

def ori_to_color(ori_deg):
    """Map orientation (0-180°) to hue."""
    hue = (ori_deg % 180) / 180
    return hsv_to_rgb([hue, 0.8, 0.9])

# Get unique planes per mouse
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flat

# Only use cells with reasonable OSI
well_tuned = osi > 0.1

for mi, mouse in enumerate(mouse_ids):
    ax = axes[mi]
    m_mask_t = (obs['mouse_id'].values == mouse) & well_tuned
    x = coords[m_mask_t, 0]
    y = coords[m_mask_t, 1]
    z = coords[m_mask_t, 2]
    po = pref_ori[m_mask_t]
    
    colors = np.array([ori_to_color(o) for o in po])
    sc = ax.scatter(x, y, c=colors, s=15, alpha=0.7)
    ax.set_xlabel('X (µm)')
    ax.set_ylabel('Y (µm)')
    ax.set_title(f'Mouse {mouse} (n={m_mask_t.sum()})', fontweight='bold')
    ax.set_aspect('equal')

# Colorbar legend
ax_cb = axes[-1]
ax_cb.axis('off')
gradient = np.linspace(0, 180, 100)
colors_bar = np.array([ori_to_color(o) for o in gradient])
for i, (o, c) in enumerate(zip(gradient, colors_bar)):
    ax_cb.barh(o, 1, color=c, height=2)
ax_cb.set_ylabel('Preferred Orientation (°)')
ax_cb.set_title('Color Legend')
ax_cb.set_xlim(0, 1.5)
ax_cb.yaxis.set_visible(True)

plt.suptitle('D1: Spatial Maps of Preferred Orientation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Layer-specific analysis: L2/3 vs L4/5 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (layer, sc_name) in zip(axes, [('L2/3', '007 L2/3 IT CTX Glut'),
                                         ('L4/5', '006 L4/5 IT CTX Glut')]):
    layer_mask = (obs['subclass_name'].values == sc_name) & well_tuned
    if layer_mask.sum() < 10:
        ax.set_visible(False)
        continue
    layer_dist = pdist(coords[layer_mask])
    layer_signal = signal_corr[np.ix_(np.where(m_mask)[0][layer_mask[m_mask]],
                                       np.where(m_mask)[0][layer_mask[m_mask]])] \
                   if layer_mask.sum() > 2 else np.array([])
    
    # Simpler: compute signal corr from orientation responses within this layer
    layer_ori = ori_responses[layer_mask]
    layer_sc = np.corrcoef(layer_ori)
    layer_d = squareform(pdist(coords[layer_mask]))
    
    triu = np.triu_indices(layer_mask.sum(), k=1)
    d_vals = layer_d[triu]
    s_vals = layer_sc[triu]
    
    # Binned
    for b in range(len(distance_edges)-1):
        bm = (d_vals >= distance_edges[b]) & (d_vals < distance_edges[b+1])
        if bm.sum() > 5:
            ax.scatter(distance_centers[b], np.nanmean(s_vals[bm]), c=SUBCLASS_COLORS[sc_name], s=40)
    
    r, p = spearmanr(d_vals[~np.isnan(s_vals)], s_vals[~np.isnan(s_vals)])
    ax.set_xlabel('Distance (µm)')
    ax.set_ylabel('Signal Correlation')
    ax.set_title(f'D1: {layer} Within-Type (ρ={r:.3f}, p={p:.1e})', fontweight='bold')
    ax.axhline(0, color='gray', ls='--', alpha=0.4)

plt.tight_layout()
plt.show()